# 🌊 Week 1, Day 5 — May 23, 2026
## Outlier Spectrum Analysis — IQR Method

## Formula: `IQR = Q3 - Q1` | Fences: `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]`



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {{
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed'] + '/plastic_filtered.csv')
df_species = pd.read_csv(DIRS['processed'] + '/species_filtered.csv')
df_climate = pd.read_csv(DIRS['processed'] + '/climate_flagged.csv')
print('Datasets loaded ✅')

## Step 1: IQR Function

In [ ]:
def compute_iqr_fences(series, label=''):
  
    series = series.dropna()
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    
    outliers = series[(series < lower_fence) | (series > upper_fence)]
    
    print(f'  {label}:')
    print(f'    Q1={q1:.3f}  Q3={q3:.3f}  IQR={iqr:.3f}')
    print(f'    Lower fence: {lower_fence:.3f}  |  Upper fence: {upper_fence:.3f}')
    print(f'    Outliers: {len(outliers)} / {len(series)} ({len(outliers)/len(series)*100:.1f}%)')
    if len(outliers) > 0:
        print(f'    Sample outlier values: {sorted(outliers.unique())[:5]}')
    print()
    
    return {
        'column': label, 'q1': round(q1,4), 'q3': round(q3,4), 'iqr': round(iqr,4),
        'lower_fence': round(lower_fence,4), 'upper_fence': round(upper_fence,4),
        'n_outliers': int(len(outliers)), 'outlier_pct': round(len(outliers)/len(series)*100, 2)
    }

print('IQR function defined ✅')

## Step 2: IQR Analysis — ocean_plastic_pollution_data

In [ ]:
print('=== OCEAN PLASTIC — IQR FENCES ===\n')
plastic_fences = {}

for col in ['Plastic_Weight_kg', 'Depth_meters', 'Latitude', 'Longitude']:
    result = compute_iqr_fences(df_plastic[col], col)
    plastic_fences[col] = result

print('FINDING: No outliers in Plastic_Weight_kg or Depth_meters.')
print('This is expected — the data uses a uniform sampling design.')
print('No clipping needed for plastic numeric columns.')

## Step 3: IQR Analysis — india_cms_final_master_enriched

In [ ]:
print('=== SPECIES DATA — IQR FENCES ===\n')
species_fences = {}

for col in ['year', 'latitude', 'longitude']:
    result = compute_iqr_fences(df_species[col], col)
    species_fences[col] = result

print('KEY FINDINGS:')
print('  year: 1,685 outlier records (13.6%) — years before 2017.5')
print('  These include legitimate historical records (2003–2016)')
print('  Decision: DO NOT clip year. Instead filter to 2015–2026 in Week 2.')
print()
print('  longitude: 2,095 outlier records (16.9%)')
print('  IQR fence upper ~91.8°E — but Andaman Sea is at 93°E (valid!)')
print('  Decision: DO NOT clip longitude. Use bounding box filter instead.')

## Step 4: IQR Analysis — Ocean Climate

In [ ]:
print('=== OCEAN CLIMATE — IQR FENCES ===\n')
climate_fences = {}

for col in ['SST (°C)', 'pH Level', 'Species Observed']:
    result = compute_iqr_fences(df_climate[col], col)
    climate_fences[col] = result

## Step 5: Box Plots — Visual Outlier Inspection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_config = [
    (df_plastic, 'Plastic_Weight_kg', axes[0][0], 'Plastic Weight (kg)',   'steelblue'),
    (df_plastic, 'Depth_meters',      axes[0][1], 'Depth (meters)',        'teal'),
    (df_species, 'year',              axes[0][2], 'Species Obs. Year',     'darkorange'),
    (df_species, 'latitude',          axes[1][0], 'Species Latitude',      'green'),
    (df_species, 'longitude',         axes[1][1], 'Species Longitude',     'purple'),
    (df_climate, 'SST (°C)',          axes[1][2], 'SST °C',               'crimson'),
]

for df, col, ax, title, color in plot_config:
    data = df[col].dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='o', markersize=3, alpha=0.4))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.suptitle('May 23, 2026 — IQR Outlier Box Plots\nAll Key Numeric Columns', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] + '/Day5_IQR_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Box plots saved ✅')

## Step 6: Save IQR Fence Values to Metadata

In [ ]:
all_fences = {
    'ocean_plastic': plastic_fences,
    'india_species': species_fences,
    'ocean_climate': climate_fences
}

fence_path = DIRS['metadata'] + '/iqr_fences.json'
with open(fence_path, 'w') as f:
    json.dump(all_fences, f, indent=2)

print(f'IQR fences saved: {fence_path} ✅')
print()
print('These fence values will be loaded in Week 2 (May 28) for outlier clipping.')
print()
print('SUMMARY — Clipping decisions:')
print('  Plastic_Weight_kg : No outliers → no clipping needed')
print('  Depth_meters      : No outliers → no clipping needed')
print('  species.year      : 13.6% flagged → use FILTER 2015-2026, not IQR clip')
print('  species.longitude : 16.9% flagged → use BBOX filter, not IQR clip')
print('  SST (°C)          : Check fence and clip if needed in Week 2')